# Behavioral Pattern, notebook 4: sensitivity to the thresholds

All cut points move together by -0.10, -0.05, +0.05, +0.10 (E_mid, E_low and the others shift the same way; clipped to 0.05-0.95). Target fixed in the protocol: at most 15% of
student-sprints change pattern for a shift of 0.05.

In [1]:
import sys, warnings, json, importlib.util
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score, confusion_matrix
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)
ROOT = Path("../../..").resolve()
spec = importlib.util.spec_from_file_location("bp_rules_nb", ROOT / "rules.py")
R = importlib.util.module_from_spec(spec); spec.loader.exec_module(R)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
d = pd.read_csv(Path("../../dataset/bp_inputs.csv"))
PATTERNS = list(R.PATTERNS) + ["Balanced contributor"]
def run(thresholds, frame=None):
    frame = d if frame is None else frame
    out = []
    for r in frame.itertuples():
        co = None if pd.isna(r.Co) else float(r.Co)
        bim = None if pd.isna(r.bimodal) else int(r.bimodal)
        lc = None if pd.isna(r.late_churn) else float(r.late_churn)
        out.append(R.classify({"E": float(r.E), "C": float(r.C), "Co": co}, bim, thresholds, lc))
    return out

In [2]:
def shifted(th, delta):
    moved = {k: float(np.clip(v + delta, 0.05, 0.95)) for k, v in th.items() if k != "name" and isinstance(v, (int, float)) and not isinstance(v, bool)}
    return {**th, **moved, "name": f"{th['name']}{delta:+.2f}"}   # the on/off switches of the extensions are not thresholds and stay as they are
rows = []
for name in ("proposal", "calibrated_v1", "calibrated"):
    base = th0 = R.load_thresholds(name)
    p0 = np.array([r["persona"] for r in run(th0)])
    for delta in (-0.10, -0.05, 0.05, 0.10):
        p1 = np.array([r["persona"] for r in run(shifted(th0, delta))])
        rows.append({"thresholds": name, "shift": delta, "share changing pattern": float((p0 != p1).mean()), "patterns fired (not balanced/unclassified)": float(np.isin(p1, R.PATTERNS).mean())})
sens = pd.DataFrame(rows).round(3)
sens["meets target (<= 0.15 at +/-0.05)"] = np.where(sens["shift"].abs() == 0.05, sens["share changing pattern"] <= 0.15, np.nan)
sens.to_csv(RESULTS / "sensitivity.csv", index=False)
sens

,thresholds,shift,share changing pattern,patterns fired (not balanced/unclassified),meets target (<= 0.15 at +/-0.05)
0,proposal,-0.10,0.088,0.107,NaN
1,proposal,-0.05,0.066,0.097,1.0
2,proposal,0.05,0.075,0.163,1.0
3,proposal,0.10,0.132,0.194,NaN
4,calibrated_v1,-0.10,0.238,0.439,NaN
5,calibrated_v1,-0.05,0.135,0.335,1.0
6,calibrated_v1,0.05,0.097,0.191,1.0
7,calibrated_v1,0.10,0.185,0.147,NaN
8,calibrated,-0.10,0.245,0.577,NaN
9,calibrated,-0.05,0.138,0.476,1.0


## Reading the results

**Sensitivity** (all cut points moved together; the on/off switches of the extensions are not moved). Final calibrated set: 13.8% of student-sprints change pattern for a shift of -0.05 and 11.9% for +0.05, both within the 15% target; for -0.10 it is 24.5% and for +0.10, 21.6%. Calibrated v1 (original rules): 13.5% and 9.7%. The proposal's set changes less (6.6% and 7.5%) only because it fires so rarely.

A lower shift makes more patterns fire (the share of student-sprints with a pattern goes from 36% at +0.05 to 58% at -0.10), so the cut points matter as much as the rules; they should be revisited once instructor judgements exist.
